In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath(".."))

import torch
import matplotlib.pyplot as plt

from hamiltonian.transverse_field_ising import TransverseFieldIsing
import numpy as np
import scipy as sp

## Parameters


In [ ]:
L = 8
h = 0.25
J = 1.0
periodic = False

## Exact diagonalization


In [ ]:
hamiltonian = TransverseFieldIsing(
    system_dim=torch.tensor([L]),
    phys_params=torch.tensor([h]),
    coupling=J,
    periodic=periodic,
)

H_sparse = hamiltonian.sparse_matrix()
H_hermitian = ((H_sparse + H_sparse.conj().T) / 2).real.astype(np.float64)

eigenvalues, eigenvectors = sp.sparse.linalg.eigsh(H_hermitian, k=1, which="SA")
ground_energy = float(eigenvalues[0])
ground_state = eigenvectors[:, 0]

print(f"L = {L}, J = {J}, h = {h}, periodic = {periodic}")
print(f"ground-state energy: {ground_energy:.6f}")
print(f"ground-state energy per site: {ground_energy / L:.6f}")

## Ground state


In [ ]:
pivot = int(np.argmax(np.abs(ground_state)))
psi = ground_state * np.exp(-1j * np.angle(ground_state[pivot]))

re_part = psi.real
im_part = psi.imag
probabilities = np.abs(psi) ** 2
basis_states = np.arange(len(psi))

fig, (ax_re, ax_im, ax_prob) = plt.subplots(3, 1, figsize=(10, 6), sharex=True)

ax_re.bar(basis_states, re_part, width=1.0)
ax_re.set_ylabel(r"Re $\psi(x)$")
ax_re.set_title(f"TFIM ground state (L={L}, J={J}, h={h})")

ax_im.bar(basis_states, im_part, width=1.0)
ax_im.set_ylabel(r"Im $\psi(x)$")

ax_prob.bar(basis_states, probabilities, width=1.0)
ax_prob.set_ylabel(r"$|\psi(x)|^2$")
ax_prob.set_xlabel(r"basis state $x$ (lex order, site 0 = MSB)")

plt.tight_layout()
plt.show()